# 👋 Hello World — pi.dev + Azure OpenAI

First step in exploring [**pi**](https://github.com/earendil-works/pi) — an open-source, multi-provider LLM API and agent framework. Goal: import the library, register **Azure OpenAI**, and get one response back.

We use **`@earendil-works/pi-ai`**, the unified LLM API. This notebook targets **Azure OpenAI only**, configured from three environment variables (named `AZURE_PI_TEST_*` to avoid clashing with any pre-existing `AZURE_OPENAI_*` vars in your `.env` files):

```
AZURE_PI_TEST_ENDPOINT     # e.g. https://<resource>.services.ai.azure.com  (or .../openai/v1)
AZURE_PI_TEST_API_KEY
AZURE_PI_TEST_DEPLOYMENT   # deployment name (comma/space-separated for several)
```

> **Kernel:** **Deno** (pi.dev is an ESM-only TypeScript package; the Deno kernel runs it natively with top-level await). See `../README.md`.

## 1. Load your Azure keys from a `.env` outside the repo

`loadEnvUp()` walks **up** the directory tree from this notebook's folder and loads **every** `.env` it finds along the way, printing the keys loaded from each. Files are applied **closest-first**, and within a file the **first** occurrence of a key wins. Keep your keys on disk but **outside the git repo** (e.g. in a parent folder), so they can never be committed:

```
# ../.env  (one directory above the repo)
AZURE_PI_TEST_ENDPOINT=https://<resource>.services.ai.azure.com
AZURE_PI_TEST_API_KEY=...
AZURE_PI_TEST_DEPLOYMENT=cohere-command-a
```

In [ ]:
import { loadEnvUp } from "playground/env";

// Loads EVERY .env from this folder up to the filesystem root (closest wins),
// and prints the keys loaded from each file.
const env = await loadEnvUp();
console.log(`\n${env.files.length} .env file(s) found; ${env.loaded.length} variable(s) loaded in total.`);

In [ ]:
// Create an empty Models collection — we register only Azure OpenAI below.
import { type Context, createModels } from "@earendil-works/pi-ai";

const models = createModels();
console.log("pi-ai loaded ✔");

## 2. Register Azure OpenAI

pi.dev doesn't ship this custom Azure setup, so we register it with `createProvider()`.
We target Azure's **OpenAI-compatible `/openai/v1` surface** (derived from the endpoint's
host), which needs **no `api-version`** query param and authenticates with the key via
Bearer / `api-key` header. The `/models` Model-Inference route requires `api-version`,
which pi-ai's OpenAI client can't inject — so we deliberately avoid it.

`AZURE_PI_TEST_DEPLOYMENT` is split into one model per deployment name.

In [ ]:
// Register Azure OpenAI from the three env vars.
import { createProvider, envApiKeyAuth, type Model } from "@earendil-works/pi-ai";
import { openAICompletionsApi } from "@earendil-works/pi-ai/api/openai-completions.lazy";

const rawEndpoint = Deno.env.get("AZURE_PI_TEST_ENDPOINT");
const azureApiKey = Deno.env.get("AZURE_PI_TEST_API_KEY");
const azureDeployments = (Deno.env.get("AZURE_PI_TEST_DEPLOYMENT") ?? "")
  .split(/[,\s]+/)
  .map((s) => s.trim())
  .filter(Boolean);

if (!rawEndpoint || !azureApiKey || azureDeployments.length === 0) {
  throw new Error(
    "Missing Azure config. Set AZURE_PI_TEST_ENDPOINT, AZURE_PI_TEST_API_KEY, " +
    "and AZURE_PI_TEST_DEPLOYMENT in a .env, then re-run cell 1.",
  );
}

// Normalize any endpoint form (root, /models, /openai/v1, trailing slash) to the
// clean OpenAI-compatible v1 base: `<origin>/openai/v1`.
const azureBaseUrl = `${new URL(rawEndpoint).origin}/openai/v1`;

const azureModels: Model<"openai-completions">[] = azureDeployments.map((id) => ({
  id, // sent as the `model` field; must match your Azure deployment name
  name: `${id} (Azure OpenAI)`,
  api: "openai-completions",
  provider: "azure-openai",
  baseUrl: azureBaseUrl,
  reasoning: false,
  input: ["text"],
  cost: { input: 0, output: 0, cacheRead: 0, cacheWrite: 0 }, // unknown; not tracked here
  // These describe the model's limits; adjust per deployment.
  contextWindow: 256_000, // INPUT context window (cohere-command-a is large: ~256K tokens)
  maxTokens: 8_192, // MAX OUTPUT tokens — sent as the request cap; cohere-command-a's output limit is 8192
  // Bearer (from envApiKeyAuth) authenticates the /openai/v1 surface; the
  // `api-key` header is a harmless fallback for api-key-style routes.
  headers: { "api-key": azureApiKey },
}));

models.setProvider(createProvider({
  id: "azure-openai",
  name: "Azure OpenAI",
  baseUrl: azureBaseUrl,
  auth: { apiKey: envApiKeyAuth("Azure OpenAI key", ["AZURE_PI_TEST_API_KEY"]) },
  models: azureModels,
  api: openAICompletionsApi(),
}));

console.log(`Azure OpenAI registered → base ${azureBaseUrl}`);
console.log(`  deployments: ${azureDeployments.join(", ")}`);

In [ ]:
// Use the first Azure deployment. (Change the index to try another one.)
const modelId = azureDeployments[0];
const model = models.getModel("azure-openai", modelId);
if (!model) throw new Error(`Model azure-openai/${modelId} not found.`);

console.log(`Using azure-openai/${modelId}.`);

In [ ]:
// Hello world: send one message and print the reply.
// Note: the model can't know what client library called it (pi.dev just sends a
// standard chat-completions request), so we ask something it *can* answer. The
// proof that pi.dev worked is the transport: a 200 with stopReason "stop" + usage.
const context: Context = {
  systemPrompt: "You are a friendly assistant. Keep answers to one short sentence.",
  messages: [
    { role: "user", content: "Say hello and tell me which model you are.", timestamp: Date.now() },
  ],
};

const response = await models.completeSimple(model, context);

// --- Reply ---
const textBlocks = response.content.filter((b) => b.type === "text");
if (textBlocks.length) {
  for (const block of textBlocks) console.log("🤖", (block as { text: string }).text);
} else {
  console.log("⚠️  No text in the response (stopReason:", response.stopReason + ")");
}

// --- Diagnostics ---
console.log("\nstopReason:", response.stopReason);
if (response.errorMessage) console.log("errorMessage:", response.errorMessage);
console.log(
  `tokens: ${response.usage.input} in / ${response.usage.output} out` +
  `  |  cost: $${response.usage.cost.total.toFixed(6)}`,
);

// --- Raw response (for troubleshooting) ---
console.log("\n--- raw response ---");
console.log(JSON.stringify(response, null, 2));

## ✅ What just happened

1. `loadEnvUp()` pulled your `AZURE_PI_TEST_*` values from `.env` file(s) outside the repo.
2. `createModels()` made an empty `Models` collection.
3. `createProvider()` + `models.setProvider()` registered **Azure OpenAI** against its
   OpenAI-compatible `/openai/v1` surface.
4. `models.getModel("azure-openai", deployment)` resolved the model.
5. `models.completeSimple(model, context)` ran one request and returned message `content`
   blocks plus token/cost `usage`. On failure it sets `stopReason: "error"` and
   `errorMessage` — both printed above, along with the raw response.

## Next steps to explore

- **Streaming**: swap `completeSimple` for `models.streamSimple(model, context)` and iterate the events (`text_delta`, `toolcall_end`, …).
- **Tools**: define a `Tool` with a TypeBox schema and handle `toolCall` blocks — the foundation of an agent loop.
- **Agents**: use `@earendil-works/pi-agent-core`'s `Agent` class, which wraps the tool-calling loop for you.